# Messages API con confirmación y presupuesto

In [2]:
# 1. IMPORTACIONES

import sys
from pathlib import Path
from decimal import Decimal
from anthropic import Anthropic

# SI CONFIG.PY NO ESTA EN EL MISMO DIRECTORIO, DESCOMENTAR ESTO Y AJUSTAR LA RUTA
# # Subir dos niveles desde Ckeckpoints/dia1
# PROJECT_ROOT = Path.cwd().resolve().parents[1]

# # Añadir la raíz del proyecto al PATH de Python
# if str(PROJECT_ROOT) not in sys.path:
#     sys.path.insert(0, str(PROJECT_ROOT))

from config import *

In [4]:

# 2. CONFIGURAR CLIENTE

client = Anthropic(
    api_key=ANTHROPIC_API_KEY,
    max_retries=API_MAX_RETRIES,
    timeout=API_TIMEOUT,
)

model = CLAUDE_MODEL

# 3. PREPARAR MENSAJE

messages = [
    {
        "role": "user",
        "content": (
            "Explica en dos frases quién ejecuta "
            "una herramienta de cliente."
        ),
    }
]

max_output = min(128, MAX_OUTPUT_TOKENS)

# 4. CONTAR TOKENS

count = client.messages.count_tokens(
    model=model,
    messages=messages,
).input_tokens

# Seleccionar tarifas según el contexto
if count > LONG_CONTEXT_THRESHOLD:
    input_rate = CLAUDE_LONG_INPUT_USD_PER_M
    output_rate = CLAUDE_LONG_OUTPUT_USD_PER_M
else:
    input_rate = CLAUDE_INPUT_USD_PER_M
    output_rate = CLAUDE_OUTPUT_USD_PER_M

# Estimación conservadora del coste
reserve = (
    Decimal(count) * input_rate
    + Decimal(max_output) * output_rate
) / Decimal(1_000_000)

reserve *= COST_SAFETY_MARGIN

# 5. COMPROBAR PRESUPUESTO

remaining = Decimal(
    input("Presupuesto restante estimado en USD: ")
)

if remaining < 0:
    raise ValueError("El presupuesto no puede ser negativo.")

print(f"\nModelo: {model}")
print(f"Tokens de entrada: {count}")
print(f"Máximo de salida: {max_output}")
print(f"Reserva estimada: ${reserve:.6f}")
print(f"Límite por ejercicio: ${MAX_COST_PER_CALL}")

if reserve > MAX_COST_PER_CALL or reserve > remaining:
    raise RuntimeError(
        "La reserva supera el presupuesto permitido."
    )

# 6. AUTORIZAR LLAMADA

authorization = input(
    "\nEscribe AUTORIZO para ejecutar la llamada: "
)

if authorization != "AUTORIZO":
    raise RuntimeError("Llamada cancelada.")

# 7. EJECUTAR CLAUDE

response = client.messages.create(
    model=model,
    max_tokens=max_output,
    messages=messages,
)

# 8. MOSTRAR RESPUESTA

response_text = "\n".join(
    block.text
    for block in response.content
    if block.type == "text"
)

print("\nRESPUESTA DE CLAUDE:\n")
print(response_text)

print("\nStop reason:", response.stop_reason)

# 9. CALCULAR COSTE REAL ESTIMADO

usage = response.usage

cost = (
    Decimal(usage.input_tokens) * input_rate
    + Decimal(usage.output_tokens) * output_rate
) / Decimal(1_000_000)

print("\nTokens consumidos:")
print("Entrada:", usage.input_tokens)
print("Salida:", usage.output_tokens)

print(f"Coste estimado: ${cost:.6f}")
print(f"Presupuesto estimado restante: ${remaining - cost:.6f}")



Modelo: claude-haiku-5-5
Tokens de entrada: 33
Máximo de salida: 128
Reserva estimada: $0.000081
Límite por ejercicio: $0.03

RESPUESTA DE CLAUDE:

Una herramienta de cliente la ejecuta la aplicación cliente, es decir,

Stop reason: max_tokens

Tokens consumidos:
Entrada: 33
Salida: 128
Coste estimado: $0.000067
Presupuesto estimado restante: $0.999933
